# Validators and Computed Fields

While `Field` covers basic constraints, Pydantic allows powerful custom validation logic using decorators.

## 1. Field Validators (`@field_validator`)

Use `@field_validator` to enforce custom logic on specific fields.

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class User(BaseModel):
    username: str

    @field_validator('username')
    @classmethod
    def check_alphanumeric(cls, v: str) -> str:
        if not v.isalnum():
            raise ValueError('Username must be alphanumeric')
        return v

try:
    User(username="user_name")  # Fails due to underscore
except ValidationError as e:
    print(e)

### Validator Modes: `before` vs `after`
- `mode='after'` (Default): Runs *after* Pydantic's internal validation (e.g., type checking).
- `mode='before'`: Runs *before* Pydantic's validation. Useful for pre-processing raw data.

In [ ]:
class Price(BaseModel):
    amount: float

    @field_validator('amount', mode='before')
    @classmethod
    def parse_currency(cls, v):
        if isinstance(v, str):
            return float(v.replace('$', '').replace(',', ''))
        return v

print(Price(amount="$1,200.50"))  # Successfully converts to 1200.5

## 2. Model Validators (`@model_validator`)

Use `@model_validator` when validation depends on multiple fields (e.g., password confirmation).

In [ ]:
from pydantic import model_validator

class Signup(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode='after')
    def check_passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError('Passwords do not match')
        return self

try:
    Signup(password="secret", confirm_password="wrong")
except ValidationError as e:
    print(e)

## 3. Computed Fields (`@computed_field`)

Computed fields are derived from other fields. They are included in serialization (e.g., when converting to JSON).

In [ ]:
from pydantic import computed_field

class Rectangle(BaseModel):
    width: float
    height: float

    @computed_field
    @property
    def area(self) -> float:
        return self.width * self.height

rect = Rectangle(width=10, height=5)
print(rect.area)  # 50.0
print(rect.model_dump())  # Includes 'area': 50.0